# HRS DDL and DML Template Architecture

**"How everything fits together."**

---

## Table of Contents

1. [Document Information](#1-document-information)
2. [Objective](#2-objective)
3. [Scope](#3-scope)
   * [3.1 DDL Generation](#31-ddl-generation)
   * [3.2 DML Generation](#32-dml-generation)
4. [Architecture Overview](#4-architecture-overview)
   * [4.1 HRS DDL Specification](#41-hrs-ddl-specification)
   * [4.2 HRS DML Specification](#42-hrs-dml-specification)
5. [Architecture Relationship](#5-architecture-relationship)
6. [DDL Template](#6-ddl-template)
7. [DDL Generation Requirements](#7-ddl-generation-requirements)
8. [DML Template](#8-dml-template)
9. [DML Generation Requirements](#9-dml-generation-requirements)
10. [RAND HRS Source Structure](#10-rand-hrs-source-structure)
11. [Natural Identifier and Surrogate Key Architecture](#11-natural-identifier-and-surrogate-key-architecture)
    * [11.1 Respondent](#111-respondent)
    * [11.2 Survey Wave](#112-survey-wave)
12. [Target Business Grain](#12-target-business-grain)
13. [Subject-Area DML Specifications](#13-subject-area-dml-specifications)
14. [Source-to-Target Mapping](#14-source-to-target-mapping)
15. [Wave-Specific and Wave-Invariant Variables](#15-wave-specific-and-wave-invariant-variables)
16. [Transformation Architecture](#16-transformation-architecture)
17. [DML Design Principles](#17-dml-design-principles)
18. [Separation of Responsibilities](#18-separation-of-responsibilities)
19. [Input and Output Architecture](#19-input-and-output-architecture)
    * [19.1 DDL](#191-ddl)
    * [19.2 DML](#192-dml)
20. [Template Usage Process](#20-template-usage-process)
21. [File Organization](#21-file-organization)
22. [SQL Generation Standards](#22-sql-generation-standards)
23. [Validation Architecture](#23-validation-architecture)
    * [23.1 DDL Validation](#231-ddl-validation)
    * [23.2 DML Validation](#232-dml-validation)
24. [Specification Authority](#24-specification-authority)
25. [Architecture Summary](#25-architecture-summary)
26. [Final Design Principle](#26-final-design-principle)

---


## 1. Document Information

| Property          | Value                                 |
| ----------------- | ------------------------------------- |
| Document Name     | HRS DDL and DML Template Architecture |
| Version           | 1.0                                   |
| Author            | Perez                                 |
| AI Assistant      | ChatGPT                               |
| Last Updated      | 2026-09-08                            |
| Target Platform   | Databricks                            |
| Compute           | Serverless                            |
| Runtime           | client.5.12                           |
| Data Source       | RAND HRS Longitudinal Dataset         |
| Target Data Layer | Silver CDM                            |

---

# 2. Objective

The objective of this architecture is to provide a standardized, reusable approach for generating Databricks SQL DDL and DML scripts for the HRS Silver CDM.

The architecture separates the physical definition of a target table from the processes used to populate that table.

The architecture has two primary objectives:

1. Provide a standardized DDL specification and template for defining HRS Silver CDM tables.

2. Provide a standardized DML specification and template for transforming and loading RAND HRS source data into the corresponding Silver CDM tables.

Each HRS survey section should have its own subject-area DML specification.

The DML specification should reference the applicable target DDL specification rather than repeatedly redefining the physical structure of the target table.

This separation allows the target table structure and the data-loading logic to be developed, maintained, and validated independently.

---

# 3. Scope

This architecture covers two related SQL-generation processes:

1. **DDL Generation**
2. **DML Generation**

## 3.1 DDL Generation

DDL generation defines the physical structure of the Silver CDM target table.

The DDL specification defines:

* Catalog
* Schema
* Table name
* Table type
* Storage format
* Columns
* Data types
* Nullability
* Identity columns
* Primary keys
* Foreign keys
* Table comments
* Column comments
* Other applicable constraints

## 3.2 DML Generation

DML generation defines how RAND HRS source data is transformed and loaded into the existing Silver CDM target table.

The DML specification defines:

* Source data
* Target data
* Target business grain
* Natural identifiers
* Parent-key lookups
* Wave processing
* Source-to-target mappings
* Wide-to-long transformation
* Unpivoting
* Data-type conversions
* Business transformations
* NULL handling
* RAND HRS missing-value handling
* Audit-column population
* Duplicate handling
* Target loading
* Validation and reconciliation

---

# 4. Architecture Overview

There are two primary components in the architecture.

## 4.1 HRS DDL Specification

**Purpose:**

> What does the Silver CDM table look like?

**Template File:**

`HRS_DDL_Master_Template.ipynb`

The DDL specification is used to generate the SQL required to create the target Silver CDM table.

The DDL specification defines the physical and structural characteristics of the target table.

---

## 4.2 HRS DML Specification

**Purpose:**

> How do we populate the Silver CDM table correctly from RAND HRS?

**Template File:**

`HRS_DML_Master_Template.ipynb`

The DML specification is used to generate the SQL required to transform and load RAND HRS source data into an existing Silver CDM target table.

The DML specification depends on the target table structure defined by the corresponding DDL specification.

The DML specification should not duplicate the physical table definition contained in the DDL specification.

---

# 5. Architecture Relationship

The DDL and DML specifications have a parent-child relationship.

The DDL specification defines the target structure.

The DML specification uses that structure as the destination for transformed source data.

The relationship can be summarized as:

```text
                 HRS DDL SPECIFICATION
                          │
                          ▼
                 Creates Silver CDM
                     TARGET TABLE
                          │
              ┌───────────┼───────────┐
              ▼           ▼           ▼
        Target Schema  Constraints  Data Types
              │           │           │
              └───────────┼───────────┘
                          │
                          ▼
                    TARGET TABLE
                          │
                          │
                          ▼
                 HRS DML SPECIFICATION
                          │
              ┌───────────┼───────────┐
              ▼           ▼           ▼
           Lookups   Transformations  Load
              │           │           │
              ▼           ▼           ▼
         HHIDPN → ID  Unpivoting    INSERT
         Wave → ID    Mappings      Audit Columns
                      Type Casting
                      NULL Rules
```

The DDL establishes the destination.

The DML prepares and loads the data into that destination.

---

# 6. DDL Template

## Template File

`HRS_DDL_Master_Template.ipynb`

The DDL template is the standardized input structure used to generate the SQL DDL for an HRS Silver CDM table.

The user provides the table-specific requirements in the DDL specification.

The DDL generator then produces the corresponding Databricks SQL DDL script.

---

# 7. DDL Generation Requirements

The DDL generator must:

1. Create the specified managed Delta table.

2. Use the specified catalog, schema, and table name.

3. Create the system-generated identity primary key when specified.

4. Define all required target columns.

5. Apply the specified Databricks data types.

6. Apply the specified nullability.

7. Define the respondent foreign key when specified.

8. Define the wave foreign key when specified.

9. Define applicable constraints.

10. Include the specified table comment.

11. Include the specified column comments.

12. Follow the specified SQL formatting requirements.

13. Use only DDL statements.

14. Not generate `INSERT`, `UPDATE`, `DELETE`, `MERGE`, or other DML operations.

15. Not invent columns, constraints, transformations, or business rules that are not defined in the specification.

16. Produce the final SQL as the sole deliverable.

The DDL generator must treat the specification as the authoritative definition of the target table.

---

# 8. DML Template

## Template File

`HRS_DML_Master_Template.ipynb`

The DML template is the standardized input structure used to generate the SQL required to populate an HRS Silver CDM table.

The DML specification is intentionally separate from the DDL specification.

The DML specification does not redefine the physical table structure.

Instead, it identifies:

* Where the source data comes from.
* Which target table receives the data.
* How source records are associated with respondents.
* How source records are associated with survey waves.
* How source variables map to target columns.
* How the RAND HRS wide longitudinal structure is transformed into the Silver CDM respondent-wave structure.
* How values are converted and transformed.
* How missing values are handled.
* How the target table is populated.

---

# 9. DML Generation Requirements

The DML generator must:

1. Read the specified RAND HRS source table.

2. Use the specified target table.

3. Resolve the source respondent natural identifier (`HHIDPN`) to the system-generated `respondent_id`.

4. Resolve the source wave identifier (`wave_number`) to the system-generated `wave_id`.

5. Transform wave-specific RAND HRS variables into the target respondent-wave structure.

6. Apply the specified source-to-target mappings.

7. Apply explicitly defined data-type conversions.

8. Apply explicitly defined business transformations.

9. Apply explicitly defined NULL and missing-value rules.

10. Apply wave-invariant variables according to the specified business rules.

11. Populate required audit columns.

12. Apply the specified duplicate-handling rules.

13. Load the transformed records using the specified load pattern.

14. Validate the resulting target data.

15. Not create or modify the target table structure.

16. Not generate surrogate parent keys.

17. Not invent source mappings or business rules.

18. Produce the final SQL as the sole deliverable.

---

# 10. RAND HRS Source Structure

The RAND HRS Longitudinal Dataset is generally organized as a wide dataset.

Wave-specific variables may appear as repeated variables across survey waves.

For example:

```text
HHIDPN
R1AGEY_E
R2AGEY_E
R3AGEY_E
...
R16AGEY_E
```

The Silver CDM target is generally organized at the respondent-wave grain.

Therefore, the DML may need to transform the source from a wide structure into a long respondent-wave structure.

Conceptually:

```text
RAND HRS SOURCE
────────────────────────────────────

Respondent
    │
    ├── Wave 1 variables
    ├── Wave 2 variables
    ├── Wave 3 variables
    │
    └── Wave N variables

             │
             │ Unpivot / Transform
             ▼

SILVER CDM TARGET
────────────────────────────────────

Respondent  Wave 1
Respondent  Wave 2
Respondent  Wave 3
...
Respondent  Wave N
```

The exact transformation must be defined by the subject-area DML specification.

The generator must not infer the transformation solely from variable names.

---

# 11. Natural Identifier and Surrogate Key Architecture

The HRS Silver CDM uses system-generated surrogate keys for parent entities.

The source RAND HRS data uses natural identifiers to locate those keys.

## 11.1 Respondent

```text
HHIDPN
   │
   ▼
hub_respondent.HHIDPN
   │
   ▼
hub_respondent.respondent_id
   │
   ▼
Target.respondent_id
```

`HHIDPN` is the natural respondent identifier.

`respondent_id` is the system-generated surrogate key.

The DML must retrieve `respondent_id` from `hub_respondent`.

The DML must not generate `respondent_id`.

---

## 11.2 Survey Wave

```text
wave_number
   │
   ▼
dim_wave.wave_number
   │
   ▼
dim_wave.wave_id
   │
   ▼
Target.wave_id
```

`wave_number` is the natural wave identifier.

`wave_id` is the system-generated surrogate key.

The DML must retrieve `wave_id` from `dim_wave`.

The DML must not generate `wave_id`.

---

# 12. Target Business Grain

Unless otherwise specified by a subject-area specification:

> One target row represents one respondent for one survey wave.

The logical business grain is:

```text
respondent_id + wave_id
```

The natural source representation is:

```text
HHIDPN + wave_number
```

The DML transformation therefore converts:

```text
HHIDPN + wave_number
```

into:

```text
respondent_id + wave_id
```

The generated DML must preserve this business grain.

---

# 13. Subject-Area DML Specifications

Each HRS survey section should have its own subject-area DML specification.

Examples include:

```text
HRS Demographics DML Specification
HRS Health DML Specification
HRS Financial DML Specification
HRS Employment DML Specification
HRS Family Structure DML Specification
```

Each subject-area specification should reference the applicable target DDL specification.

The subject-area specification supplies the information that changes between HRS survey sections.

This includes:

* Source variables
* Target columns
* Wave applicability
* Transformations
* Missing-value rules
* Business rules
* Filters
* Validation requirements

The master DML template supplies the common technical structure.

---

# 14. Source-to-Target Mapping

The source-to-target mapping matrix is a mandatory component of each subject-area DML specification.

The mapping identifies how each source variable is transformed into a target column.

The recommended structure is:

| Wave     | Source Variable     | Variable Label | RAND Type     | Target Column     | Databricks Type | Transformation     | Nullable | Missing-Value Rule |
| -------- | ------------------- | -------------- | ------------- | ----------------- | --------------- | ------------------ | -------- | ------------------ |
| `<wave>` | `<source_variable>` | `<label>`      | `<RAND type>` | `<target_column>` | `<target type>` | `<transformation>` | Yes/No   | `<rule>`           |

The DML generator must use the mapping matrix as the authoritative source for source-to-target transformations.

---

# 15. Wave-Specific and Wave-Invariant Variables

Each subject-area specification must identify whether a source variable is:

1. Wave-specific, or
2. Wave-invariant.

### Wave-Specific Example

```text
R1AGEY_E → Wave 1 → agey_e
R2AGEY_E → Wave 2 → agey_e
R3AGEY_E → Wave 3 → agey_e
```

### Wave-Invariant Example

```text
RARACEM  → raracem
RAHISPAN → rahispan
RAEDYRS  → raedyrs
RARELIG  → rarelig
RAVETRN  → ravetrn
```

The subject-area specification must define how wave-invariant values are associated with respondent-wave observations.

The DML generator must not infer this behavior.

---

# 16. Transformation Architecture

The DML transformation process generally consists of the following stages:

```text
SOURCE DATA
     │
     ▼
Source Filtering
     │
     ▼
Wave Transformation
     │
     ▼
Source-to-Target Mapping
     │
     ▼
Data-Type Conversion
     │
     ▼
Respondent Key Resolution
     │
     ▼
Wave Key Resolution
     │
     ▼
Business Rules
     │
     ▼
NULL / Missing-Value Rules
     │
     ▼
Audit Columns
     │
     ▼
Business-Grain Validation
     │
     ▼
TARGET INSERT
```

The actual SQL implementation may use CTEs, joins, unpivot operations, or other supported SQL techniques.

---

# 17. DML Design Principles

The following principles apply to all DML generated from this architecture.

### 17.1 Do Not Invent Mappings

If a source-to-target mapping is not provided, the generator must not invent one.

### 17.2 Do Not Invent Business Rules

If the specification does not define how a value should be interpreted or transformed, the generator must not infer the business rule.

### 17.3 Do Not Generate Parent Surrogate Keys

Parent surrogate keys must always be resolved from their parent tables.

### 17.4 Do Not Populate Identity Columns

System-generated identity columns must be excluded from the target `INSERT` column list.

### 17.5 Preserve NULL

NULL values must remain NULL unless an explicit transformation rule states otherwise.

### 17.6 Preserve Target Grain

The DML must not produce duplicate observations for the defined target business grain.

### 17.7 Do Not Silently Discard Records

Records that cannot be resolved or transformed correctly should be identifiable for investigation.

---

# 18. Separation of Responsibilities

The architecture separates responsibilities between DDL and DML.

| Responsibility             | DDL |         DML         |
| -------------------------- | :-: | :-----------------: |
| Define catalog             |  ✓  |      Reference      |
| Define schema              |  ✓  |      Reference      |
| Define table               |  ✓  |      Reference      |
| Define columns             |  ✓  |      Reference      |
| Define data types          |  ✓  |      Reference      |
| Define identity key        |  ✓  |  Exclude from load  |
| Define primary key         |  ✓  |      Reference      |
| Define foreign keys        |  ✓  | Resolve parent keys |
| Define source table        |  —  |          ✓          |
| Define source mappings     |  —  |          ✓          |
| Define transformations     |  —  |          ✓          |
| Define wave processing     |  —  |          ✓          |
| Define missing-value rules |  —  |          ✓          |
| Populate audit columns     |  —  |          ✓          |
| Load target data           |  —  |          ✓          |
| Validate loaded data       |  —  |          ✓          |

---

# 19. Input and Output Architecture

## 19.1 DDL

```text
DDL Specification
       │
       ▼
HRS_DDL_Master_Template.ipynb
       │
       ▼
Generated Databricks SQL
       │
       ▼
CREATE TARGET TABLE
```

## 19.2 DML

```text
DML Specification
       │
       ▼
HRS_DML_Master_Template.ipynb
       │
       ▼
Generated Databricks SQL
       │
       ▼
TRANSFORM + INSERT DATA
```

---

# 20. Template Usage Process

The recommended development process is:

### Step 1 — Define the Target Table

Create the subject-area DDL specification.

### Step 2 — Generate DDL

Use:

`HRS_DDL_Master_Template.ipynb`

to generate the target table DDL.

### Step 3 — Review and Validate DDL

Confirm:

* Table name
* Columns
* Data types
* Keys
* Constraints
* Comments
* Nullability

### Step 4 — Create the Target Table

Execute the generated DDL in the appropriate Databricks environment.

### Step 5 — Create the Subject-Area DML Specification

Define:

* Source variables
* Target mappings
* Waves
* Transformations
* Parent-key lookups
* Missing-value rules
* Business rules

### Step 6 — Generate DML

Use:

`HRS_DML_Master_Template.ipynb`

to generate the DML.

### Step 7 — Validate DML

Validate:

* Parent-key resolution
* Wave processing
* Source-to-target mappings
* Data types
* Business grain
* Missing-value handling
* Record counts

### Step 8 — Execute DML

Load the transformed data into the existing Silver CDM table.

---

# 21. File Organization

The generated SQL should follow the HRS repository structure.

```text
/sql
   │
   ├── ddl
   │     ├── create_<table_1>.sql
   │     ├── create_<table_2>.sql
   │     └── ...
   │
   └── dml
         ├── load_<table_1>.sql
         ├── load_<table_2>.sql
         └── ...
```

The DDL and DML scripts should have matching target table names.

For example:

```text
create_fact_demographics.sql
load_fact_demographics.sql
```

---

# 22. SQL Generation Standards

All generated SQL must:

* Be compatible with Databricks SQL.
* Use fully qualified table names where required.
* Use uppercase SQL keywords.
* Use consistent indentation.
* Use explicit column lists.
* Use descriptive aliases.
* Use descriptive CTE names when CTEs are used.
* Avoid unnecessary SQL complexity.
* Follow the applicable DDL or DML specification exactly.

---

# 23. Validation Architecture

Validation occurs at two levels.

## 23.1 DDL Validation

DDL validation confirms that the target table was created according to the DDL specification.

Examples include:

* Table exists
* Correct catalog
* Correct schema
* Correct table name
* Delta format
* Managed table
* Required columns
* Correct data types
* Identity column
* Primary key
* Foreign keys
* Comments

## 23.2 DML Validation

DML validation confirms that source data was transformed and loaded correctly.

Examples include:

* Source records identified
* Respondent keys resolved
* Wave keys resolved
* Expected waves present
* Source mappings applied
* Data types converted correctly
* NULL rules applied
* Missing-value rules applied
* No duplicate respondent-wave observations
* Expected record counts
* Source-to-target reconciliation

---

# 24. Specification Authority

The applicable subject-area specification is the authoritative source for business and transformation rules.

The master templates provide the standardized structure and technical rules.

The SQL generator must follow the following hierarchy:

```text
Subject-Area Specification
          │
          ▼
Master Template
          │
          ▼
Generated SQL
```

The generator must not replace explicit specification requirements with assumptions.

When required information is missing or ambiguous, the generator should identify the missing requirement rather than silently inventing a solution.

---

# 25. Architecture Summary

The HRS DDL and DML architecture establishes a repeatable process for building the Silver CDM.

The architecture separates:

**DDL**

> Defines what the target table looks like.

from:

**DML**

> Defines how the target table is populated from RAND HRS.

The DDL establishes the target structure.

The DML uses that structure to transform the wide RAND HRS source data into the respondent-wave Silver CDM structure.

The overall architecture is:

```text
                 RAND HRS SOURCE
                        │
                        │
                        ▼
             ┌──────────────────────┐
             │  HRS DML SPECIFICATION │
             └──────────────────────┘
                        │
          ┌─────────────┼─────────────┐
          ▼             ▼             ▼
       Lookups     Transformations    Load
          │             │             │
          ▼             ▼             ▼
      HHIDPN → ID   Wide → Long      INSERT
      Wave → ID     Unpivoting       Audit
                    Mappings
                    Casting
                    NULL Rules
                        │
                        ▼
              ┌──────────────────────┐
              │   SILVER CDM TABLE   │
              └──────────────────────┘
                        ▲
                        │
              ┌──────────────────────┐
              │  HRS DDL SPECIFICATION │
              └──────────────────────┘
                        │
                        ▼
                 Defines Table
```

The architecture is designed to support consistent, maintainable, and repeatable SQL generation across all HRS Silver CDM subject areas.

---

# 26. Final Design Principle

The HRS DDL and DML templates are **generation frameworks**, not substitutes for business knowledge.

The user supplying a subject-area specification is responsible for defining the source variables, target mappings, transformations, and business rules.

The SQL generator is responsible for translating those explicitly defined requirements into technically correct Databricks SQL.

The generator must never silently invent a business rule, source mapping, transformation, missing-value treatment, or parent-key relationship that is not explicitly defined in the applicable specification.

---

# "How do all of these pieces fit together?"
```text
HRS DDL_DML Master Template Architecture
│
├── 1. Purpose and Objectives
├── 2. Scope
├── 3. Architecture
├── 4. Component Relationships
├── 5. Overall Process Flow  ← ADD THE DIAGRAM HERE
│
├── 6. DDL Architecture
│      └── DDL Master Template
│
├── 7. DML Architecture
│      └── DML Master Template
│
└── 8. DDL-to-DML Handoff

```

```text

HRS DDL_DML Master Template Architecture
                │
       ┌────────┴────────┐
       ▼                 ▼
HRS DDL Workflow    HRS DML Workflow
       │                 │
       ▼                 ▼
DDL Master           DML Master
Template             Template
       │                 │
       ▼                 ▼
Subject DDL          Subject DML
Specification        Specification
       │                 │
       ▼                 ▼
Generated DDL        Generated DML
       │                 │
       ▼                 ▼
Delta Table          Loaded Data
```
